# HackAIthon 2026 — Bảng C


## 1. Cài thư viện

In [1]:

!pip install -q rank_bm25 pandas tqdm huggingface_hub
!pip install -q llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


try:
    import llama_cpp
    print("llama-cpp-python version:", llama_cpp.__version__)
except Exception as e:
    print("LỖI:", e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 528.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
llama-cpp-python version: 0.3.31


## 2. Dynamic Auto-Loader

In [2]:
import os, json, pandas as pd
from pathlib import Path
import glob, re
from huggingface_hub import snapshot_download

# --- CẤU HÌNH ---
MODEL_REPO = "Jackrong/Qwen3.5-4B-Claude-4.6-Opus-Reasoning-Distilled-v2-GGUF"
GGUF_GLOB = "*Q5_K_M*.gguf"
LOAD_LOCAL = False  # Đổi thành True khi Submit Offline

OUT_DIR = Path("/kaggle/working")
MODEL_DIR = OUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def find_file(pattern):
    if Path("/kaggle/input").exists():
        files = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
        return Path(files[0]) if files else None
    return None

def find_gguf(directory, pattern):
    files = list(Path(directory).rglob(pattern.replace("*", "*")))
    return files[0] if files else None

TEST_PATH = find_file("*test*.json")
KB_PATH = find_file("*vietnam_kb.jsonl")

if not TEST_PATH: raise FileNotFoundError("❌ LỖI: Không tìm thấy file đề thi!")

if LOAD_LOCAL:
    MODEL_PATH = find_gguf("/kaggle/input", GGUF_GLOB)
else:
    MODEL_PATH = find_gguf(MODEL_DIR, GGUF_GLOB)
    if not MODEL_PATH:
        snapshot_download(repo_id=MODEL_REPO, allow_patterns=GGUF_GLOB, local_dir=MODEL_DIR, local_dir_use_symlinks=False)
        MODEL_PATH = find_gguf(MODEL_DIR, GGUF_GLOB)

print(f"Model : {MODEL_PATH}")
print(f"KB    : {KB_PATH}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Model : /kaggle/working/models/Qwen3.5-4B.Q5_K_M.gguf
KB    : /kaggle/input/datasets/kimductri/hackaithon/vietnam_kb.jsonl


## 3. Khởi tạo Offline RAG (BM25) & Engine Llama.cpp

In [3]:
from rank_bm25 import BM25Okapi
import json
import re

STOP_WORDS = set(["của", "là", "và", "các", "những", "thì", "mà", "theo", "quy", "định",
                  "nào", "sau", "đây", "trong", "có", "không", "được", "với", "cho",
                  "câu", "hỏi", "hãy", "chọn", "đáp", "án", "một", "này", "đó",
                  "như", "về", "từ", "đến", "khi", "tại", "trên", "dưới", "nếu",
                  "vì", "do", "bởi", "cũng", "đều", "rằng", "thế", "rất", "hay"])

def create_ngrams(text, n=2):
    words = [w for w in text.lower().split() if w not in STOP_WORDS and len(w) > 1]
    ngrams = words[:]
    for i in range(len(words) - 1):
        ngrams.append("_".join(words[i:i+2]))
    return ngrams

corpus = []
bm25 = None

CHUNK_SIZE = 150
STEP_SIZE = 100

if KB_PATH and KB_PATH.exists():
    with open(KB_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            doc = json.loads(line)
            title = doc.get('title', '')
            content = doc.get('content', '')
            words = content.split()

            if len(words) <= CHUNK_SIZE:
                corpus.append(f"[{title}]\n{content}")
            else:
                for i in range(0, len(words), STEP_SIZE):
                    chunk_words = words[i:i + CHUNK_SIZE]
                    if len(chunk_words) < 30: break
                    corpus.append(f"[{title}]\n{' '.join(chunk_words)}")

    bm25 = BM25Okapi([create_ngrams(doc, n=2) for doc in corpus])
    print(f"Đã nạp và tối ưu {len(corpus)} phân đoạn ngữ nghĩa (Overlapping).")

def offline_search(question, choices):
    if not bm25 or not corpus: return None

    # LỌC RÁC: Loại bỏ các lựa chọn vô nghĩa để BM25 không bị nhiễu
    noise_kws = ["tất cả", "cả a", "cả b", "cả c", "đều", "từ chối", "không có", "chưa rõ", "không thể", "đáp án", "phương án"]
    clean_choices = []
    for c in choices[:4]:
        c_str = str(c).lower()
        if not any(kw in c_str for kw in noise_kws):
            clean_choices.append(str(c))
            
    choice_text = " ".join(clean_choices)  
    raw_query = re.sub(r"[^\w\s]", " ", (question + " " + choice_text).lower())
    clean_words = create_ngrams(raw_query, n=2)

    if not clean_words: clean_words = question.lower().split()

    top_10_docs = bm25.get_top_n(clean_words, corpus, n=10)
    scored_docs = []
    choice_strings = [str(c).lower().strip() for c in choices]

    for doc in top_10_docs:
        doc_lower = doc.lower()
        score = 0
        for c_str in choice_strings:
            if len(c_str) > 3 and c_str in doc_lower:
                score += 10
        scored_docs.append((score, doc))

    scored_docs.sort(key=lambda x: x[0], reverse=True)
    best_docs = [doc for score, doc in scored_docs[:3]]
    if not best_docs: return None

    combined_context = "\n---\n".join(best_docs)
    
    # Ép an toàn 600 từ để không tràn RAM
    final_words = combined_context.split()
    if len(final_words) > 600:
        combined_context = " ".join(final_words[:600]) + "...\n[Đã cắt bớt]"

    return combined_context

Đã nạp và tối ưu 7309 phân đoạn ngữ nghĩa (Overlapping).


## 4. Khởi tạo model


In [4]:
from llama_cpp import Llama

try:
    llm = Llama(
        model_path=str(MODEL_PATH),
        n_gpu_layers=-1, 
        n_ctx=4096,  # Tối ưu (để chứa đủ 1 chunk RAG)
        verbose=False
    )
    print("KHỞI TẠO LLM THÀNH CÔNG!")
except Exception as e:
    raise RuntimeError(f"❌ LỖI: {e}")

llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


KHỞI TẠO LLM THÀNH CÔNG!


## 5. Promt Build

In [5]:
import re

LET = "ABCDEFGHIJK"

def has_embedded_context(question):
    markers = ["Đoạn thông tin", "Tiêu đề:", "Nội dung:", "Đoạn văn:", "Dựa vào đoạn", "Cho đoạn", "Đọc đoạn", "Xét đoạn"]
    return any(m in question for m in markers)

def check_ethics_override(question, choices):
    q_str = str(question).lower()
    toxic_action_kws = ["trái quy định", "đình chỉ", "tránh việc cung cấp", "phát tán", "làm suy yếu", "chống lại", "xúc phạm", "làm giả", "tham nhũng", "phá hoại", "kích động", "vi phạm", "đánh cắp", "lừa đảo", "trốn", "lạm dụng", "lách luật", "hack", "xâm nhập"]
    if any(kw in q_str for kw in toxic_action_kws):
        refusal_kws = ["tôi không thể", "từ chối", "không thể trả lời", "vi phạm pháp luật", "không cung cấp", "không được phép", "xin lỗi"]
        if choices:
            for i, c in enumerate(choices):
                if any(kw in str(c).lower() for kw in refusal_kws): return LET[i]
    return None

def route_question(question, choices):
    text_lower = question.lower()
    scores = {"CALCULATION": 0, "VIETNAM": 0, "HISTORY_LAW": 0, "GENERAL": 0}
    
    hard_math_kws = ["đạo hàm", "tích phân", "ma trận", "xác suất", "gdp", "lãi suất", "phương trình", "đồ thị", "hàm số", "tiệm cận", "logarit", "thể tích", "diện tích", "chu vi", "cấp số cộng", "cấp số nhân", "gia tốc", "động năng", "thế năng", "bước sóng"]
    scores["CALCULATION"] += sum(3 for kw in hard_math_kws if kw in text_lower)
    if re.search(r"\btính\b(?!\s+(chất|cách|từ|mạng|nhân|dục|kế))", text_lower): scores["CALCULATION"] += 1
    if re.search(r"\bgiá trị\b(?!\s+(nhân đạo|lịch sử|văn hóa|nghệ thuật|đạo đức|tinh thần))", text_lower): scores["CALCULATION"] += 1
    if re.search(r"∫|∑|lim|log|sin|cos|tan|√|\d+\s*[\+\-\/=]\s*\d+|[<>]\s*\d+", text_lower): scores["CALCULATION"] += 2
        
    his_kws = ["chiến tranh", "triều đại", "hiệp ước", "hiệp định", "thế kỷ", "cổ đại", "phong kiến", "lịch sử", "khởi nghĩa", "kháng chiến", "thực dân", "đế quốc", "vua", "hoàng đế", "công ước", "pháp luật"]
    scores["HISTORY_LAW"] += sum(2 for kw in his_kws if kw in text_lower)
    if re.search(r"\bnăm \d{3,4}\b", text_lower): scores["HISTORY_LAW"] += 3

    vn_kws = ["việt nam", "đảng", "hồ chí minh", "hiến pháp", "nghị định", "thông tư", "tư tưởng", "mác - lênin", "chính trị", "cách mạng", "nhà nước", "chính phủ", "quốc hội", "ban chấp hành", "trung ương", "điều luật", 
          "văn học", "tục ngữ", "thành ngữ", "ca dao","thơ","truyện","tác phẩm", "địa lý", "tỉnh", "thành phố", "đồng bằng"]
    scores["VIETNAM"] += sum(2 for kw in vn_kws if kw in text_lower)
    
    if choices:
        num_numeric = sum(1 for c in choices if re.match(r"^[\d\.\,\-\+\/\\\spi\%]+$", str(c).strip()))
        if num_numeric >= len(choices) / 2:
            num_years = sum(1 for c in choices if re.match(r"^(1|2)\d{3}$", str(c).strip()))
            if num_years == num_numeric and ("năm" in text_lower or "thế kỷ" in text_lower): scores["HISTORY_LAW"] += 5
            else: scores["CALCULATION"] += 4

    cat = max(scores, key=scores.get)
    return "GENERAL" if scores[cat] == 0 else cat

def format_choices(ch): return "\n".join(f"{LET[i]}. {c}" for i, c in enumerate(ch))

# KHÔI PHỤC PROMPT CÂN BẰNG: Cho phép mô hình suy luận vừa đủ, không làm biếng
PROFILES = {
    "CALCULATION": "Bạn là AI Toán học. HƯỚNG DẪN TƯ DUY: 1. Ghi công thức. 2. Tính toán cẩn thận từng bước. 3. Đối chiếu và chốt đáp án.",
    "VIETNAM": "Bạn là AI Pháp lý Việt Nam. Rút trích từ khóa và đối chiếu với điều luật để chọn đáp án.",
    "HISTORY_LAW": "Bạn là AI Lịch sử Thế giới. So sánh các mốc thời gian và sự kiện.",
    "GENERAL": "Bạn là AI Logic. Dùng phương pháp loại trừ, đánh giá đúng/sai từng lựa chọn trong tối đa 3 câu."
}

def build_prompt(x, category, context=None):
    valid_letters = "{" + ", ".join(list(LET[:len(x['choices'])])) + "}"
    
    if has_embedded_context(x['question']):
        ctx_str = "LỆNH: Đọc đoạn thông tin có sẵn trong đề bài để trả lời. Không cần dùng kiến thức ngoài.\n"
    elif context:
        ctx_str = f"[TÀI LIỆU SỰ THẬT]:\n{context}\n\nLỆNH QUAN TRỌNG: Nếu tài liệu chứa đáp án, hãy dùng làm gốc. NẾU TÀI LIỆU KHÔNG CHỨA THÔNG TIN, BẠN BẮT BUỘC PHẢI DÙNG KIẾN THỨC CỦA CHÍNH MÌNH ĐỂ TRẢ LỜI ĐÚNG NHẤT.\n"
    else:
        ctx_str = ""
        
    body = f"{ctx_str}\nCâu hỏi: {x['question']}\n\nCác lựa chọn:\n{format_choices(x['choices'])}\n\n"
    instruction = f"""{PROFILES.get(category, PROFILES["GENERAL"])}\n\nLỆNH BẮT BUỘC TUÂN THỦ:\n1. Trả lời hoàn toàn bằng Tiếng Việt.\n2. Suy nghĩ trong thẻ <think> BẮT BUỘC ngắn gọn, rõ ràng.\n3. Ngay sau khi đóng thẻ </think>, BẮT BUỘC ghi duy nhất 1 dòng: ĐÁP ÁN: X (Với X thuộc {valid_letters}).\n\n<think>\n"""
    return body + instruction

def parse_marker(text, n):
    valid = set(list(LET[:n]))
    clean_text = re.sub(r"<think>.*?</think>", " ", text or "", flags=re.DOTALL)
    patterns = [r"(?:ĐÁP ÁN|Đáp án|Lựa chọn|Chọn|Đáp án là)[\s:=>\-\[\]]*([A-K])\b", r"Đáp án[:\s]*([A-K])\b", r"Chọn[:\s]*([A-K])\b"]
    for pat in patterns:
        matches = list(re.finditer(pat, clean_text, re.I))
        if matches:
            last_match = matches[-1].group(1).upper()
            if last_match in valid: return last_match
    for pat in patterns:
        matches = list(re.finditer(pat, text or "", re.I))
        if matches:
            last_match = matches[-1].group(1).upper()
            if last_match in valid: return last_match
    return None

def fuzzy_math_matcher(raw_thought, choices, valid_letters):
    numbers_in_thought = re.findall(r"[-+]?\d*\.\d+|\d+", raw_thought)
    if not numbers_in_thought: return None
    final_number_str = numbers_in_thought[-1].replace(",", "")
    for i, c in enumerate(choices):
        if i >= len(valid_letters): break
        c_str = str(c).replace(",", "")
        c_numbers = re.findall(r"[-+]?\d*\.\d+|\d+", c_str)
        if final_number_str in c_numbers: return valid_letters[i]
        try:
            f_val = float(final_number_str)
            for cn in c_numbers:
                c_val = float(cn)
                if c_val != 0 and abs(f_val - c_val) / abs(c_val) < 0.02: return valid_letters[i]
        except ValueError: continue
    return None

def smart_fallback(question, choices, context_used):
    combined = (question + " " + (context_used or "")).lower()
    best_idx, best_score = 0, -1
    for i, c in enumerate(choices):
        c_words = [w for w in str(c).lower().split() if len(w) > 2 and w not in STOP_WORDS]
        score = sum(1 for w in c_words if w in combined)
        if score > best_score:
            best_score, best_idx = score, i
    return LET[best_idx]

## 6. 2 pass verify

In [6]:
import time
import os
import glob
import json
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter

def chat(messages, max_tokens, temp=0.1):
    out = llm.create_chat_completion(messages=messages, max_tokens=max_tokens, temperature=temp, top_p=0.9)
    return out["choices"][0]["message"]["content"].strip()

def answer_one(x):
    n = len(x["choices"])
    valid_letters = list(LET[:n])
    
    ethics_letter = check_ethics_override(x["question"], x["choices"])
    if ethics_letter: return ethics_letter, "hardcode_ETHICS", "Hard-coded Ethics Override."

    category = route_question(x["question"], x["choices"])
    context = None
    if not has_embedded_context(x["question"]) and category in ["VIETNAM", "HISTORY_LAW"]:
        context = offline_search(x["question"], x["choices"]) 
        
    # NỚI LỎNG TOKEN (Giúp khắc phục hoàn toàn 18 lỗi Fallback của bản trước)
    if category == "CALCULATION": max_t = 1024 
    elif category == "ETHICS": max_t = 300
    else: max_t = 800
    
    prompt = build_prompt(x, category, context)
    messages_pass_1 = [{"role": "system", "content": "Bạn là AI thông minh."}, {"role": "user", "content": prompt}]
    raw1 = chat(messages_pass_1, max_tokens=max_t, temp=0.2)
    letter = parse_marker(raw1, n)
    
    src = f"rag_{category}" if context else f"gen_{category}"
    final_raw = raw1
    context_for_fallback = context
    
    if letter is None and category == "CALCULATION":
        letter = fuzzy_math_matcher(raw1, x['choices'], valid_letters)
        if letter:
            src = f"gen_CALCULATION_fuzzy_match"
            return letter, src, raw1[-200:]

    if letter is None:
        partial_thought = raw1[-1500:] if len(raw1) > 1500 else raw1
        messages_pass_2 = [
            {"role": "system", "content": "Bạn là AI thông minh."},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": partial_thought},
            {"role": "user", "content": f"BẮT BUỘC chỉ trả lời đúng MỘT dòng: 'ĐÁP ÁN: X' (với X thuộc {valid_letters}). KHÔNG dùng thẻ <think>."}
        ]
        raw2 = chat(messages_pass_2, max_tokens=150, temp=0.0)
        letter = parse_marker(raw2, n)
        if letter:
            src = src + "_conclude"
            final_raw = raw1 + "\n\n[ÉP CHỐT]:\n" + raw2
        else:
            final_raw = raw1 + "\n\n[LƯỢT 2 THẤT BẠI]:\n" + raw2
            
    if letter is None:
        safe_tail = final_raw.replace("[ÉP CHỐT]", "").replace("[LƯỢT 2 THẤT BẠI]", "")
        for pat in [r"(?:đáp án|chọn|là|kết quả)[\s:=>\-\[\]]*([A-K])\b"]:
            for m in reversed(list(re.finditer(pat, safe_tail[-150:].lower()))):
                found = m.group(1).upper()
                if found in set(valid_letters): 
                    letter, src = found, src + "_safe_loose"
                    break
        if letter is None:
            letter = smart_fallback(x["question"], x["choices"], context_for_fallback)
            src = src + "_smart_fallback"
            
    return letter, src, final_raw[-200:]

with open(TEST_PATH, 'r', encoding='utf-8') as f:
    questions = json.load(f)

BATCH_SIZE = 50
t0 = time.time()
total_rows = 0
all_srcs = []

print(f"BẮT ĐẦU GIẢI {len(questions)} CÂU HỎI...")
TEMP_DIR = OUT_DIR / "temp_batches"
TEMP_DIR.mkdir(exist_ok=True)

for i in range(0, len(questions), BATCH_SIZE):
    batch = questions[i:i + BATCH_SIZE]
    batch_rows = [] 
    batch_idx = i // BATCH_SIZE + 1
    for x in tqdm(batch, desc=f"Batch {batch_idx}"):
        llm._ctx.kv_cache_clear() 
        letter, src, tail = answer_one(x)
        batch_rows.append({"qid": x["qid"], "answer": letter, "src": src, "raw_tail": tail})
        all_srcs.append(src) 
    
    df_batch = pd.DataFrame(batch_rows)
    df_batch[['qid', 'answer']].to_csv(TEMP_DIR / f"pred_batch_{batch_idx}.csv", index=False)
    df_batch.to_csv(TEMP_DIR / f"details_batch_{batch_idx}.csv", index=False, encoding='utf-8-sig')
    
    total_rows += len(batch_rows)
    print(f"Đã lưu Batch {batch_idx} ({total_rows}/{len(questions)} câu)")
    del batch_rows, df_batch

def extract_batch_num(filename):
    match = re.search(r'batch_(\d+)\.csv', filename)
    return int(match.group(1)) if match else 0

all_pred_files = sorted(glob.glob(str(TEMP_DIR / "pred_batch_*.csv")), key=extract_batch_num)
all_details_files = sorted(glob.glob(str(TEMP_DIR / "details_batch_*.csv")), key=extract_batch_num)

df_final_pred = pd.concat((pd.read_csv(f) for f in all_pred_files), ignore_index=True)
df_final_details = pd.concat((pd.read_csv(f) for f in all_details_files), ignore_index=True)

df_final_pred.to_csv(OUT_DIR / "pred.csv", index=False)
df_final_details.to_csv(OUT_DIR / "prediction_details.csv", index=False, encoding='utf-8-sig')

for f in all_pred_files + all_details_files:
    os.remove(f)
os.rmdir(TEMP_DIR)

dt = time.time() - t0
print(f"Tổng thời gian: {dt:.0f}s ({dt/max(total_rows,1):.1f}s/câu)")
print(f"Nguồn phân phối: {dict(Counter(all_srcs))}")

BẮT ĐẦU GIẢI 463 CÂU HỎI...


Batch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Đã lưu Batch 1 (50/463 câu)


Batch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Đã lưu Batch 2 (100/463 câu)


...

Đã lưu Batch 8 (400/463 câu)


Batch 9:   0%|          | 0/50 [00:00<?, ?it/s]

Đã lưu Batch 9 (450/463 câu)


Batch 10:   0%|          | 0/13 [00:00<?, ?it/s]

Đã lưu Batch 10 (463/463 câu)
Tổng thời gian: 4321s (9.3s/câu)
Nguồn phân phối: {'gen_HISTORY_LAW': 46, 'gen_CALCULATION': 153, 'gen_GENERAL': 141, 'gen_GENERAL_smart_fallback': 7, 'rag_VIETNAM': 56, 'hardcode_ETHICS': 12, 'gen_VIETNAM': 31, 'rag_HISTORY_LAW': 5, 'rag_VIETNAM_conclude': 1, 'gen_CALCULATION_fuzzy_match': 4, 'gen_CALCULATION_conclude': 1, 'rag_HISTORY_LAW_conclude': 2, 'gen_GENERAL_conclude': 2, 'gen_VIETNAM_smart_fallback': 1, 'gen_CALCULATION_smart_fallback': 1}
